# Ollama Legal PDF RAG Notebook

## Import Libraries


In [1]:
# Imports
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain.retrievers.multi_query import MultiQueryRetriever

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Jupyter-specific imports
from IPython.display import display, Markdown

# Set environment variable for protobuf
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

## Load PDF
change the path to document for different case files

In [2]:
# Load PDF

local_path = "sample_data\Ram_Sarup_Gupta_Dead_By_Lrs_vs_Bishun_Narain_Inter_College_Ors_on_8_April_1987.PDF"
if local_path:
    loader = UnstructuredPDFLoader(file_path=local_path)
    data = loader.load()
    print(f"PDF loaded successfully: {local_path}")
else:
    print("Upload a PDF file")

PDF loaded successfully: sample_data\Ram_Sarup_Gupta_Dead_By_Lrs_vs_Bishun_Narain_Inter_College_Ors_on_8_April_1987.PDF


## Split text into chunks

In [3]:
# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=200)
chunks = text_splitter.split_documents(data)
print(f"Text split into {len(chunks)} chunks")

Text split into 31 chunks


In [4]:
#print(chunks)  # Display first 500 characters of the first chunk

## Create vector database

In [5]:
# Create vector database
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=OllamaEmbeddings(model="nomic-embed-text"),
    collection_name="local-rag"
)
print("Vector database created successfully")

Vector database created successfully


## Set up LLM and Retrieval

In [6]:
# Set up LLM and retrieval
local_model = "gemma3:4b"  # or whichever model you prefer
llm = ChatOllama(model=local_model)

In [20]:
# Query prompt template
QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to generate 2
    different versions of the given user question to retrieve relevant documents from
    a vector database. By generating multiple perspectives on the user question, your
    goal is to help the user overcome some of the limitations of the distance-based
    similarity search. Provide these alternative questions separated by newlines.
    Original question: {question}""",
)

# Set up retriever
retriever = MultiQueryRetriever.from_llm(
    vector_db.as_retriever(), 
    llm,
    prompt=QUERY_PROMPT
)

## Create chain

In [8]:
# RAG prompt template
template = """
You are a legal assistant. Use ONLY the information provided in the context below to answer the question as accurately and concisely as possible. Do not make up any information or provide legal advice.

Context:
{context}

Question:
{question}

Answer (based strictly on the context):
"""

prompt = ChatPromptTemplate.from_template(template)

In [9]:
# Create chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## Chat with PDF

In [10]:
def chat_with_pdf(question):
    """
    Chat with the PDF using the RAG chain.
    """
    return display(Markdown(chain.invoke(question)))

In [12]:
chat_with_pdf("What is the main idea of Irrevocable License for School: Ram Sarup Gupta vs Bishun Narain?")

The core argument in *Ram Sarup Gupta vs. Bishun Narain* revolves around whether the school could establish an irrevocable license due to the construction of permanent buildings. The court ultimately ruled against the school. Here’s a breakdown of the key points:

*   **Section 60(b) Requirement:** The school argued that constructing permanent buildings and incurring expenses under the license made it irrevocable (as per section 60(b) of the Act).
*   **Lack of Proper Pleadings:** The court found that the school failed to adequately plead that the construction and expenses were made “acting upon the license.” Specifically, they didn’t present sufficient evidence to demonstrate they were performing the license’s terms when building.
*   **No Objection from Co-Sharers:** Crucially, the court noted that no one involved in the joint family (including the co-sharers) raised objections to the license, implying acquiescence and further weakening the school’s case.

In essence, the court determined that the school didn't meet the legal requirements to establish an irrevocable license simply by building permanent structures. The crucial element missing was a clear demonstration that they were operating under the terms of the license when undertaking those construction activities.

In [ ]:
chat_with_pdf("Summarize the arguments regarding the irrevocability of the license.")

The core argument revolves around whether the school’s construction on the property constituted a permanent, irrevocable license under Section 60(b) of the Indian Easements Act. Here’s a breakdown of the key arguments:

**School’s Argument (for Irrevocability):**

*   **Permanent Construction:** The school erected works of a permanent nature (buildings) on the property.
*   **“Acting Upon the License”:** They did so while utilizing the license for the purpose of running the school.
*   **Expenses Incurred:** They spent money on these constructions as part of operating the school.

**Opposing Argument (against Irrevocability):**

*   **Lack of Specific Pleadings:** The opposing side argued the school failed to present sufficient pleadings to demonstrate the license was truly irrevocable. They contended that the courts shouldn't have created a new case for the defendants.
*   **Absence of Issue Framed:** No specific issue was framed concerning the irrevocability of the license.

**Court’s Decision (Based on the provided text):**

The court ultimately ruled in favor of the school, stating that all three conditions – permanent construction, "acting upon the license," and incurred expenses – were met, making the license irrevocable. The court emphasized that allowing someone to build on land while granting a license doesn’t automatically create a right to revoke it later. 

**In essence, the arguments hinged on whether the school’s actions fully satisfied the requirements for an irrevocable license under Section 60(b) of the Easements Act.**

In [ ]:
chat_with_pdf("")Can you explain the case study highlighted in the document?

Okay, let's break down the case study presented in the document.

**The Case:** The document outlines a legal dispute concerning the revocation of a license granted to a school (the “respondents”) by Raja Ram Kumar Bhargava (the “grantor”). The school had been running a business on land leased from Bhargava.

**Key Arguments & Findings:**

1. **Bhargava’s Claim:** Bhargava argued that the license was irrevocable under two clauses of Section 60 of the Act (which deals with irrevocable licenses):
   * **(a) Transfer of Property:** He claimed the license was linked to a transfer of property (renting out land to others).
   * **(b) Permanent Work:** He asserted that the school had built permanent structures and incurred expenses while operating under the license.

2. **Court’s Decision:**  Both the trial court and the High Court rejected Bhargava’s claim that the license was irrevocable under clause (a). However, they *did* uphold Bhargava’s argument that the license *was* irrevocable under clause (b). 

3. **Lack of Explicit Wording:** Crucially, the document states that the written statement by Bhargava didn’t explicitly use the phrase “executed work of permanent character.” However, the court inferred this from the overall content of the statement. It concluded that Bhargava had intended to raise this argument.

4. **Focus on Inference:** The core of the case rested on the court’s ability to *infer* that Bhargava had intended to claim that the school had invested in permanent structures due to the circumstances described in his written statement.

**In essence, the case highlights that even if a pleading doesn't use exact wording, a court can still find that the relevant argument was raised, particularly when the circumstances clearly point to the intent to invoke a specific legal provision.**

Do you want me to elaborate on any particular aspect of this case (e.g., the legal provisions involved, the reasoning behind the court’s decision, or the significance of the case)?

In [15]:
chat_with_pdf("Why was the school seeking recognition from the Education Department, and how did the original owner's action help secure this?")

The school was seeking recognition from the Education Department to establish its legitimacy and obtain operational permissions. The original owner, Raja Ram Kumar Bhargava, helped secure this recognition by permanently donating the property to the school. He named the school after his father, Bishun Narain Bhargava, and the Managing Committee of the school subsequently expressed deep gratitude and appreciation to him for this donation, explicitly resolving to name the school in his father’s memory. This act of donation, along with the school's subsequent need for additional buildings to provide classrooms, was seen as evidence of the school’s purpose and enabled it to obtain recognition from the U.P. Government.

## Chatbot UI

In [13]:
import gradio as gr
import nest_asyncio
nest_asyncio.apply()

def gradio_interface(question, history):
    try:
        response = chain.invoke(question)
    except Exception:
        response = "⚠️ Sorry, something went wrong while processing your request."
    return response

# Gradio UI
demo = gr.ChatInterface(
    fn=gradio_interface,
    title="Legal Document Chatbot",
    description="Ask questions about the Document",
    examples=[
        "tell me the key points in the document.",
        "Summarize the court's reasoning on this case."
    ]
)

# ⚠️ Safely launch Gradio without showing tracebacks
try:
    demo.launch(share=True, show_error=False)  # `show_error=False` hides internal errors
except Exception:
    pass  


* Running on local URL:  http://127.0.0.1:7860


## Clean up (optional)

In [17]:
# Optional: Clean up when done 
#vector_db.delete_collection()
#print("Vector database deleted successfully")